# Getting started with krovlab

`krovlab.roof` turns a building footprint and a pitch into a roof you can read quantities off.

This notebook walks through:

1. Calling the single entry point
2. Reading faces, arcs and heights
3. Checking that the roof is a valid terrain
4. A rectangle (one ridge) versus a square (an apex)
5. Pitch as degrees, rise:run or a percentage
6. What happens when the input cannot be roofed
7. A plan view of the skeleton

Units are metres and degrees throughout. The core library has no third-party dependencies; Plotly is used only at the end to draw.

In [ ]:
from krovlab import Failure, Roof, Validity, roof

## A square hip roof

A 10 m square at 45° is the smallest interesting case. The four hips meet at an apex whose height is half the side times `tan(pitch)`: `5 × tan(45°) = 5 m`.

Pass the footprint as a list of `(x, y)` corners. Either winding is fine; do not repeat the first point at the end.

In [ ]:
square = [(0.0, 0.0), (10.0, 0.0), (10.0, 10.0), (0.0, 10.0)]
square_roof = roof(square, 45.0)

assert isinstance(square_roof, Roof)
print(f"ridge height:      {square_roof.ridge_height:.3f} m")
print(f"total sloped area: {square_roof.total_sloped_area:.3f} m²")
print(f"terrain:           {square_roof.validity.is_terrain}")
print(f"nodes: {len(square_roof.nodes)}, faces: {len(square_roof.faces)}, arcs: {len(square_roof.arcs)}")

### Faces

Each face rises from one footprint edge. `edge_index` `i` is the edge from `footprint[i]` to `footprint[(i + 1) % n]`.

`plan_area` is the horizontal projection (the four faces of this square sum to 100 m²). `sloped_area` is what covering is bought by: `plan_area / cos(pitch)`.

In [ ]:
print(f"{'edge':>4}  {'pitch':>6}  {'plan m²':>8}  {'sloped m²':>10}")
for face in square_roof.faces:
    print(
        f"{face.edge_index:4d}  {face.pitch:6.1f}  "
        f"{face.plan_area:8.3f}  {face.sloped_area:10.3f}"
    )
print(f"{'sum':>4}  {'':>6}  {sum(f.plan_area for f in square_roof.faces):8.3f}  "
      f"{square_roof.total_sloped_area:10.3f}")

### Arcs

Every named line of the roof is an `Arc` with a 3D `length` in metres:

- **eave** — the footprint edge (gutter)
- **hip** — rising from a convex corner
- **ridge** — horizontal, both ends above the eave (none on a square: the top is a point)

In [ ]:
from collections import defaultdict

by_kind: dict[str, list[float]] = defaultdict(list)
for arc in square_roof.arcs:
    by_kind[arc.kind].append(arc.length)

for kind, lengths in by_kind.items():
    total = sum(lengths)
    print(f"{kind:5s}  {len(lengths)} run(s)  total {total:.3f} m  {lengths}")

### Node heights

Footprint corners sit at height 0. The apex is the wavefront event that closed the square — that event's *time* is the height; there is no separate lifting step.

In [ ]:
for i, node in enumerate(square_roof.nodes):
    print(f"node {i}: ({node.x:5.2f}, {node.y:5.2f})  height {node.height:.3f} m")

### Validity

Every roof is checked when it is built. `validity.is_terrain` is true when the faces cover the footprint once, each face is planar, sampled plan points have one height, and water on each face runs to that face's own eave — not merely to whichever eave is nearest.

If a check fails, `validity.reasons` names the property that broke. You read it off the value; you do not have to rerun the tests.

In [ ]:
assert isinstance(square_roof.validity, Validity)
print(f"is_terrain: {square_roof.validity.is_terrain}")
print(f"reasons:    {square_roof.validity.reasons}")

## A rectangle: one ridge

A 10 × 6 m rectangle at 45° is not a pyramid. The two short sides collapse first, leaving a ridge of length `10 − 6 = 4 m` at height `3 × tan(45°) = 3 m`.

In [ ]:
rectangle = [(0.0, 0.0), (10.0, 0.0), (10.0, 6.0), (0.0, 6.0)]
rect_roof = roof(rectangle, 45.0)
assert isinstance(rect_roof, Roof)

ridges = [arc for arc in rect_roof.arcs if arc.kind == "ridge"]
print(f"ridge height: {rect_roof.ridge_height:.3f} m")
print(f"ridge count:  {len(ridges)}")
for arc in ridges:
    a, b = rect_roof.nodes[arc.start], rect_roof.nodes[arc.end]
    print(
        f"ridge {arc.length:.3f} m  "
        f"({a.x:.1f}, {a.y:.1f}, {a.height:.1f}) → "
        f"({b.x:.1f}, {b.y:.1f}, {b.height:.1f})"
    )

## Pitch spellings

The same slope can be written as degrees, a rise:run ratio, or a percentage. `100%` is `1:1` is `45°` — `tan(45°) = 1`. The roof is the same.

In [ ]:
for spelling in (45.0, (1, 1), "1:1", "100%"):
    result = roof(square, spelling)
    assert isinstance(result, Roof)
    print(f"{spelling!r:>8}  ridge {result.ridge_height:.3f} m  face pitch {result.faces[0].pitch}°")

## When the input cannot be roofed

`Failure` is a value, not an exception. Branch on `kind`; show `reason` to a person. A folder of footprints can keep going past the bad one.

In [ ]:
for label, footprint, pitch in (
    ("out of range", square, 0.0),
    ("wrong list length", square, [45.0, 45.0]),
    ("bowtie", [(0, 0), (10, 10), (10, 0), (0, 10)], 45.0),
    ("a line", [(0, 0), (10, 0), (4, 0)], 45.0),
    ("clockwise square", list(reversed(square)), 45.0),
):
    result = roof(footprint, pitch)
    if isinstance(result, Failure):
        print(f"{label:20s}  {result.kind:20s}  {result.reason}")
    else:
        print(f"{label:20s}  Roof                 ridge {result.ridge_height:.3f} m")

## Plan view of the skeleton

The roof is data. A plan view is just a consumer of `nodes` and `arcs`: eaves in grey, hips in orange, the ridge in red. Heights are labelled at each node.

Needs Plotly (`uv sync --extra viz`, or the `dev` group).

In [ ]:
import plotly.graph_objects as go

COLOUR = {"eave": "#6b7280", "hip": "#d97706", "ridge": "#dc2626"}


def plan_figure(built: Roof, title: str) -> go.Figure:
    fig = go.Figure()
    drawn: set[str] = set()
    for arc in built.arcs:
        a, b = built.nodes[arc.start], built.nodes[arc.end]
        show = arc.kind if arc.kind not in drawn else None
        drawn.add(arc.kind)
        fig.add_trace(
            go.Scatter(
                x=[a.x, b.x],
                y=[a.y, b.y],
                mode="lines",
                line={"color": COLOUR[arc.kind], "width": 3 if arc.kind != "eave" else 2},
                name=arc.kind,
                legendgroup=arc.kind,
                showlegend=show is not None,
                hovertemplate=f"{arc.kind} {arc.length:.2f} m<extra></extra>",
            )
        )
    fig.add_trace(
        go.Scatter(
            x=[n.x for n in built.nodes],
            y=[n.y for n in built.nodes],
            text=[f"{n.height:.2f} m" for n in built.nodes],
            mode="markers+text",
            textposition="top center",
            marker={"size": 8, "color": "#111827"},
            name="nodes",
            hovertemplate="(%{x:.2f}, %{y:.2f}) h=%{text}<extra></extra>",
        )
    )
    fig.update_layout(
        title=title,
        xaxis_title="x (m)",
        yaxis_title="y (m)",
        yaxis_scaleanchor="x",
        yaxis_scaleratio=1,
        template="plotly_white",
        legend_title="arc",
        width=700,
        height=500,
    )
    return fig


plan_figure(rect_roof, "10 × 6 m rectangle at 45°").show()

The same helper on the square: four hips, no ridge, apex in the middle.

In [ ]:
plan_figure(square_roof, "10 m square at 45°").show()

## What is not here yet

The entry point now accepts either winding, three pitch spellings, and a pitch list of matching length. Bad input comes back as a `Failure` with a `kind`.

This release still only **roofs** convex footprints at **one** pitch. Reflex corners (L, T, U), differing per-edge pitches, holes, gable ends and overhang will come in later work. A valid hole or a mixed pitch list is `unsupported`, not a crash.